In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

from rdkit.Chem import Descriptors, MolFromSmiles, rdFingerprintGenerator as fp, Draw, MolToSmiles
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import Chem, RDLogger
from chython import smiles

import pandas as pd
import numpy as np
import os
import random

import warnings
warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')

In [6]:
train_data = pd.read_csv('data/final_train_data80.csv', index_col=0)
test_data = pd.read_csv('data/final_test_data80.csv', index_col=0)

In [7]:
train_data.head()

,SMILES,LogP
ID,,
0,C1(NON=C1C2=CC=CC=C2)=N,3.093
1,C=1C=CC=CC=1CC(NC=2C=CC(=CC=2)Br)=O,5.245
2,C(C)C1=CC=CS1,4.294
3,N1C=CNC1=NC2C(OC)=NC(=NC2Cl)C,2.254
4,CC(C)CCO,1.939


In [8]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13403 entries, 0 to 13402
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   SMILES  13403 non-null  object 
 1   LogP    13403 non-null  float64
dtypes: float64(1), object(1)
memory usage: 314.1+ KB


In [9]:
METAL_LIST = ['Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Ga', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Cd', 'In', 'Sn',
              'La', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'Ac', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu',
              'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No',
              'Lr', 'Ge', 'Sb', 'Na', 'Ca']


def RemoveMetalls(mol_):
    """
    Check for metals in the molecule and return None if found, 
    otherwise, return the original molecule
    """
    
    return mol_ if not any(a.GetSymbol() in METAL_LIST for a in mol_.GetAtoms()) else None


def is_salt_by_chyton(mol):
    """
    Функция для проверки, является ли молекула солью на основе наличия ионов.
    Возвращает True, если молекула содержит ионы (катионы и анионы).
    """
    
    if smiles(Chem.MolToSmiles(mol)).neutralize():
        return None
    return mol

In [10]:
def canonicalize_smiles(smiles):
    '''
    Стандартизировать SMILES с помощью различных методов библиотеки rdkit
    '''
    
    try:
        smiles, idx = smiles['SMILES'], smiles.index
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        def apply_and_count(method_name, func):
            """Вспомогательная функция для подсчёта количества изменений"""
            
            initial_smiles = Chem.MolToSmiles(mol)
            new_mol = func()
            if isinstance(new_mol, Chem.Mol):
                updated_smiles = Chem.MolToSmiles(new_mol)
            else:
                updated_smiles = Chem.MolToSmiles(mol)

            if initial_smiles != updated_smiles:
                method_counts[method_name] += 1
            if isinstance(new_mol, Chem.Mol):
                return new_mol
            return mol

        mol = apply_and_count("RemoveMetalls", lambda: RemoveMetalls(mol))
        mol = apply_and_count("RemoveSalt", lambda: is_salt_by_chyton(mol))
        mol = apply_and_count("SanitizeMol", lambda: Chem.SanitizeMol(mol))
        mol = apply_and_count('IsotopeParent', lambda: rdMolStandardize.IsotopeParent(mol))
        mol = apply_and_count('StereoParent', lambda: rdMolStandardize.StereoParent(mol))
        mol = apply_and_count('RemoveHs', lambda: Chem.RemoveHs(mol))
        mol = apply_and_count('FragmentParent', lambda: rdMolStandardize.FragmentParent(mol))
        mol = apply_and_count('Reionize', lambda: rdMolStandardize.Reionize(mol))
        mol = apply_and_count('ChargeParent', lambda: rdMolStandardize.ChargeParent(mol))
        normalizer = rdMolStandardize.Normalizer()
        mol = apply_and_count('Normalizer', lambda: normalizer.normalize(mol))
        tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
        mol = apply_and_count('TautomerCanonicalize', lambda: tautomer_enumerator.Canonicalize(mol))
        mol = apply_and_count('SetAromaticity', lambda: Chem.rdmolops.SetAromaticity(mol))

        canonical_smiles = Chem.MolToSmiles(mol)
        return canonical_smiles

    except Exception as e:
        return None

In [11]:
method_counts = {
    "RemoveSalt": 0,
    "RemoveMetalls": 0,
    "SanitizeMol": 0,
    "Kekulize": 0,
    "RemoveHs": 0,
    "IsotopeParent": 0,
    "StereoParent": 0,
    "FragmentParent": 0,
    "ChargeParent": 0,
    "Normalizer": 0,
    "TautomerCanonicalize": 0,
    "MetalDisconnector": 0,
    "Reionize": 0,
    "SetAromaticity": 0,
    "AssignStereochemistry": 0
}

In [13]:
train_data['Smiles_cleaned'] = train_data.apply(canonicalize_smiles, axis=1)